# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
%pip install "gymnasium[toy-text]"

In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.

# Respuesta

**Respuesta:**

Taxi tiene **500 estados y 6 acciones**.

Cada estado combina la posición del taxi, la ubicación del pasajero y su destino. Hay 25 posiciones del taxi, 5 ubicaciones posibles del pasajero —incluyendo estar dentro del taxi— y 4 destinos:

\[
25 \times 5 \times 4 = 500
\]

Tiene más estados que FrozenLake porque no basta con representar una casilla: también necesitamos conocer dónde está el pasajero y a dónde debe llevarlo el taxi.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


In [ ]:
nombres_acciones = {
    0: "South — abajo",
    1: "North — arriba",
    2: "East — derecha",
    3: "West — izquierda",
    4: "Pickup — recoger",
    5: "Dropoff — dejar"
}

print("Experiencia (s_t, a_t, r_t+1, s_t+1):")
print((state, action, reward, next_state))
print("Acción:", nombres_acciones[int(action)])

### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

**Interpretación:**

**Respuesta:**

La experiencia observada fue:

$$(s_t, a_t, r_{t+1}, s_{t+1}) = (386, 2, -1, 386)$$

- **Estado inicial (386):** estado del entorno antes de realizar la acción.
- **Acción (2):** mover el taxi hacia el este (East).
- **Recompensa (-1):** penalización recibida por realizar el movimiento.
- **Siguiente estado (386):** el taxi permaneció en el mismo estado. El movimiento quedó bloqueado por una pared o un límite del mapa.

Esta experiencia permite actualizar Q(386, 2), utilizando la recompensa recibida y el valor futuro estimado del siguiente estado.

## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$


**Respuesta:**

La Q-table contiene un valor por cada combinación de estado y acción. Como Taxi tiene **500 estados** y **6 acciones**, calculamos:

$$
|\mathcal{S}| \times |\mathcal{A}| = 500 \times 6 = 3000
$$

Por lo tanto, el agente tiene **3000 valores Q para estimar**. Inicialmente todos son cero y se actualizan al visitar los pares estado–acción durante el entrenamiento. Tener 3000 entradas no garantiza que todas se aprendan con la misma precisión.


## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # Explorar: seleccionar una acción aleatoria.
    if random.random() < epsilon:
        return int(env.action_space.sample())

    # Explotar: seleccionar una acción con el mayor valor Q.
    q_values = Q[state]
    max_q = np.max(q_values)

    # Si hay empate, elegir aleatoriamente entre las mejores.
    best_actions = np.flatnonzero(q_values == max_q)

    return int(np.random.choice(best_actions))

## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(
    Q,
    state,
    action,
    reward,
    next_state,
    alpha,
    gamma,
    terminated=False
):
    current_q = Q[state, action]

    # Si se alcanzó un terminal real, no hay recompensa futura.
    best_next_q = (
        0.0 if terminated else np.max(Q[next_state])
    )

    # Objetivo de aprendizaje.
    td_target = reward + gamma * best_next_q

    # Diferencia entre el objetivo y la estimación actual.
    td_error = td_target - current_q

    # Actualización de Q.
    Q[state, action] = current_q + alpha * td_error

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            # 1. Escoger una acción.
            action = choose_action(Q, state, epsilon, env)

            # 2. Ejecutar la acción.
            next_state, reward, terminated, truncated, _ = (
                env.step(action)
            )

            # 3. Actualizar el valor Q.
            update_q(
                Q,
                state,
                action,
                reward,
                next_state,
                alpha,
                gamma,
                terminated=terminated
            )

            # 4. Actualizar estado y recompensa acumulada.
            state = next_state
            total_reward += reward

            # 5. Terminar el episodio si corresponde.
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

episodios = np.arange(window, len(rewards) + 1)

plt.figure(figsize=(10, 4))
plt.plot(episodios, moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(
    f"{env.spec.id} — recompensa promedio "
    f"({window} episodios)"
)
plt.grid(alpha=0.3)
plt.show()

print(
    "Promedio de los primeros 100 episodios:",
    np.mean(rewards[:100])
)

print(
    "Promedio de los últimos 100 episodios:",
    np.mean(rewards[-100:])
)

### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?


**Respuesta:**

La recompensa promedio mejora claramente durante el entrenamiento. En los primeros 100 episodios fue de **−311.38**, mientras que en los últimos 100 alcanzó **2.32**. Esto indica que el agente aprende a tomar decisiones que reducen las penalizaciones y mejoran su recompensa acumulada.

La mejora es especialmente rápida durante los primeros 1000 episodios. A partir de aproximadamente **1500 a 2000 episodios**, la curva comienza a estabilizarse alrededor de valores cercanos a cero y ligeramente positivos, con pequeñas fluctuaciones.

Estas variaciones pueden deberse a los diferentes estados iniciales y a la exploración, porque epsilon permanece en **0.1**. La gráfica sugiere que el agente aprendió una política razonablemente buena, pero no demuestra una convergencia perfecta de todos los valores Q ni garantiza que siempre elija el recorrido óptimo.

## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


In [ ]:
print("SIN ENTRENAMIENTO")
print("Recompensa:", reward_initial)
print("Pasos:", len(frames_initial) - 1)

print("\nCON ENTRENAMIENTO")
print("Recompensa:", reward_trained)
print("Pasos:", len(frames_trained) - 1)

### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

**Respuesta:**

El taxi sin entrenamiento selecciona acciones al azar, porque todos sus valores Q son cero y el código desempata aleatoriamente. En esta ejecución obtuvo una recompensa total de **−149 durante 50 pasos**, sin completar la entrega. Este resultado refleja las penalizaciones acumuladas por sus movimientos y por acciones incorrectas de recoger o dejar al pasajero.

El agente entrenado selecciona las acciones con mayor valor Q. En esta ejecución logró recoger al pasajero y llevarlo a su destino, obteniendo una recompensa total de **10 en 11 pasos**. Este resultado es consistente con una entrega exitosa y diez acciones previas con costo de −1, sin penalizaciones adicionales por recoger o dejar incorrectamente al pasajero.

La evidencia de aprendizaje está en que el agente entrenado completó la tarea en pocos pasos y obtuvo una recompensa mucho mayor. Esto indica que aprendió a elegir acciones más útiles y a evitar errores. Sin embargo, un solo episodio no permite asegurar que siempre complete la tarea ni que todos sus recorridos sean óptimos.

La comparación tiene una limitación: el episodio sin entrenamiento permite 50 pasos y el entrenado permite 200. Aun así, el agente entrenado solo necesitó 11 pasos, una cantidad inferior a ambos límites.

## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

nombres = [
    "South — abajo",
    "North — arriba",
    "East — derecha",
    "West — izquierda",
    "Pickup — recoger",
    "Dropoff — dejar"
]

print("Estado seleccionado:", state)
print("\nValores Q:")

for codigo, nombre in enumerate(nombres):
    print(
        f"{codigo}: {nombre} = "
        f"{Q_trained[state, codigo]:.6f}"
    )

max_q = np.max(Q_trained[state])
best_actions = np.flatnonzero(
    Q_trained[state] == max_q
)

print("\nAcción o acciones con mayor valor Q:")
for codigo in best_actions:
    print(f"{codigo}: {nombres[int(codigo)]}")

### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

**Respuesta:**

1. Para el estado **123**, la acción con mayor valor Q es **3: West — izquierda**, con un valor de **3.949478**. Como no hay empate en el máximo, la política greedy seleccionaría esta acción.

2. Una acción con mayor valor Q tiene una mejor estimación de recompensa acumulada y descontada. En este estado, ir hacia la izquierda es la alternativa que el agente considera más conveniente según lo aprendido, porque su valor Q es superior al de las otras cinco acciones.

3. Q(s,a) no representa únicamente la recompensa inmediata, porque su actualización también considera las recompensas futuras estimadas desde el siguiente estado. Por ejemplo, moverse hacia la izquierda puede tener un costo inmediato de **−1**, pero su valor Q es positivo porque puede ayudar al taxi a avanzar hacia una entrega exitosa y obtener una recompensa posterior.


## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.

**Respuesta:**

Modifiqué únicamente **epsilon, de 0.1 a 0.3**. Conservé alpha en **0.1**, gamma en **0.95** y los **5000 episodios**. Ambos entrenamientos comenzaron con una Q-table en cero.

**Velocidad de aprendizaje:** la curva del entrenamiento original, con epsilon igual a 0.1, alcanza antes recompensas promedio más altas. Con epsilon igual a 0.3, la recompensa también mejora, pero permanece por debajo de la original. Esto muestra que explorar más no garantiza una mejora más rápida de la recompensa durante el entrenamiento.

**Estabilidad:** la curva con epsilon igual a 0.3 presenta fluctuaciones más marcadas. Aunque ambas curvas se estabilizan aproximadamente entre los episodios 1500 y 2000, el experimento mantiene una recompensa promedio menor. Esto es coherente con una mayor exploración, que introduce más acciones aleatorias incluso cuando el agente ya conoce buenas alternativas.

**Recompensa final:** el promedio de los últimos 100 episodios fue de **2.32** en el entrenamiento original y de **−13.48** en el experimento. Estos resultados corresponden al entrenamiento, durante el cual ambos agentes siguen explorando.

**Comportamiento observado:** al ejecutar la política greedy del experimento, el taxi completó la entrega con una recompensa de **10 en 11 pasos**, igual que el agente original en el episodio comparado. Por lo tanto, una recompensa de entrenamiento menor no significa necesariamente que la política aprendida sea peor: al dejar de explorar, el agente del experimento también logró completar la tarea.

En esta comparación, aumentar epsilon redujo la recompensa promedio durante el entrenamiento, pero ambos agentes obtuvieron el mismo resultado en el episodio greedy observado. Como los entrenamientos incluyen decisiones aleatorias y solo se comparó un episodio final, no podemos asegurar que sus políticas tengan el mismo desempeño en todos los estados.


In [ ]:
# Entrenamiento nuevo desde cero.
# Solo cambia epsilon: de 0.1 a 0.3.

Q_experiment, rewards_experiment = train_q_learning(
    env,
    np.zeros((n_states, n_actions)),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.3
)

window = 100

average_original = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

average_experiment = np.convolve(
    rewards_experiment,
    np.ones(window) / window,
    mode="valid"
)

episodios = np.arange(window, len(rewards) + 1)

plt.figure(figsize=(10, 4))
plt.plot(
    episodios,
    average_original,
    label="Original: epsilon = 0.1"
)
plt.plot(
    episodios,
    average_experiment,
    label="Experimento: epsilon = 0.3"
)

plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title("Taxi — comparación de exploración")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(
    "Promedio final del entrenamiento original:",
    np.mean(rewards[-100:])
)

print(
    "Promedio final del experimento:",
    np.mean(rewards_experiment[-100:])
)

In [ ]:
frames_experiment, reward_experiment = play_episode(
    env,
    Q_experiment,
    max_steps=200,
    seed=7
)

print("Recompensa del episodio:", reward_experiment)
print("Pasos:", len(frames_experiment) - 1)

frames_to_video(frames_experiment, interval=500)

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
